# Tiny Transformer Training

- Module: 10 Transformers and LLM Foundations
- Topic: causal language-model training with a tiny transformer
- Estimated runtime: 2-4 minutes on CPU
- Prerequisites: PyTorch basics, cross-entropy loss, causal masking, next-token prediction
- Extra dependency: `torch` must be available in the active Python environment
- Expected outputs: loss curve, gradient-norm plot, short generated sample


## Learning goals

By the end of this notebook you should be able to:

- build a tiny decoder-only transformer for character-level language modeling;
- train it with a causal next-token objective;
- inspect training loss and gradient norms; and
- generate text from the trained model to connect objective and behavior.


In [ ]:
from __future__ import annotations

import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd().parents[2]
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from shared.src.training_diagnostics import append_gradient_snapshot, plot_gradient_norms, plot_loss_curves

SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
device


## Tiny corpus and tokenization

We deliberately use a very small repeated corpus so the training loop stays fast and the objective remains easy to inspect.
This is not a production recipe. It is a compact teaching example.


In [ ]:
text = (
    "transformers attend over tokens and predict the next symbol. "
    "small models can still show the mechanics of causal training. "
) * 24

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: idx for idx, ch in enumerate(chars)}
itos = {idx: ch for ch, idx in stoi.items()}
encoded = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

context_length = 16
batch_size = 16


def decode(indices: torch.Tensor) -> str:
    return "".join(itos[int(i)] for i in indices)


def get_batch() -> tuple[torch.Tensor, torch.Tensor]:
    starts = torch.randint(0, len(encoded) - context_length - 1, (batch_size,))
    x = torch.stack([encoded[i : i + context_length] for i in starts])
    y = torch.stack([encoded[i + 1 : i + context_length + 1] for i in starts])
    return x.to(device), y.to(device)


print("Corpus length:", len(encoded))
print("Vocab size:", vocab_size)
print("Example slice:", decode(encoded[:80]))


## A minimal decoder-only transformer

The model uses token embeddings, learned positional embeddings, one transformer encoder block with a causal mask, and a linear language-model head.
Using `nn.TransformerEncoder` keeps the training notebook short while preserving the core causal objective.


In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 32, nhead: int = 4, ff_width: int = 64, num_layers: int = 1, context_length: int = 16):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context_length, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_width,
            dropout=0.0,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor, targets: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device)
        h = self.token_embedding(x) + self.position_embedding(positions)[None, :, :]
        causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=x.device), diagonal=1)
        h = self.encoder(h, mask=causal_mask)
        logits = self.head(self.ln(h))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, start_tokens: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        self.eval()
        tokens = start_tokens
        for _ in range(max_new_tokens):
            x = tokens[:, -self.context_length :]
            logits, _ = self(x)
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            tokens = torch.cat([tokens, next_token], dim=1)
        return tokens


model = TinyTransformerLM(vocab_size=vocab_size, context_length=context_length).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
sum(parameter.numel() for parameter in model.parameters())


## Training loop

We optimize the causal next-token objective for a small number of steps and record both loss and gradient norms.
The gradient diagnostics come from the shared repo utilities used elsewhere in the curriculum.


In [ ]:
steps = 80
loss_history = []
grad_history: dict[str, list[float]] = {"step": [], "total_norm": []}

model.train()
for step in range(1, steps + 1):
    xb, yb = get_batch()
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(xb, yb)
    loss.backward()
    if step % 10 == 0:
        append_gradient_snapshot(
            grad_history,
            [("token_embedding", model.token_embedding.weight), ("head", model.head.weight)],
            step=step,
        )
    optimizer.step()
    loss_history.append(float(loss.item()))

loss_history[-5:]


## Diagnostics and generation

A decreasing training loss means the model is fitting the local token statistics of the corpus.
The generated text is still crude, but it should begin to echo repeated patterns from the tiny dataset.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_loss_curves(loss_history, steps=list(range(1, len(loss_history) + 1)), ax=axes[0], title="Tiny Transformer Training Loss")
plot_gradient_norms(grad_history, ax=axes[1], title="Tracked Gradient Norms")
fig.tight_layout()
plt.show()

start = torch.tensor([[stoi["t"]]], dtype=torch.long, device=device)
sample = model.generate(start, max_new_tokens=80)[0].cpu()
print(decode(sample))


## Summary

This notebook trained a very small causal transformer on a toy corpus, tracked its optimization behavior, and generated text from the resulting model. The example is intentionally small, but it mirrors the same structural recipe used in large decoder-only language models.

### Exercises

- Replace greedy decoding with temperature sampling.
- Compare one-layer and two-layer models at the same token budget.
- Swap learned positional embeddings for sinusoidal embeddings and compare the training curves.
